In [1]:
#import required classes and packages
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from keras.models import Sequential, Model
from keras.callbacks import ModelCheckpoint
import os
import pickle
from keras.utils.np_utils import to_categorical
from keras.layers import  MaxPooling2D
from keras.layers import Dense, Dropout, Activation, Flatten, GlobalAveragePooling2D, BatchNormalization
from keras.layers import Convolution2D
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn import metrics 
import warnings
warnings.filterwarnings('ignore')

Using TensorFlow backend.


ModuleNotFoundError: No module named 'tensorflow.python'

In [ ]:
#loading and displaying cricket winner dataset values
dataset = pd.read_csv("Dataset/ball_by_ball_it20.csv", nrows=30000)
dataset

In [ ]:
#visualizing graph of successfully chased or not
Y = dataset['Chased Successfully'].ravel()
names, count = np.unique(Y, return_counts = True)
labels = ["Not Chased Successfully", 'Chased Successfully']
height = count
bars = labels
y_pos = np.arange(len(bars))
plt.figure(figsize = (4, 3)) 
plt.bar(y_pos, height)
plt.xticks(y_pos, bars)
plt.xlabel("Dataset Class Label Graph")
plt.ylabel("Count")
plt.xticks(rotation=90)
plt.show()

In [ ]:
#dataset pre-processing such as droping ir-relevant columns, replacing missing values,
#normalizing and shuffling dataset values
features = ['Chased Successfully','Target Score', 'Runs From Ball','Innings Runs', 'Innings Wickets', 'Balls Remaining', 'Target Score', 'Total Batter Runs','Total Non Striker Runs','Batter Balls Faced','Non Striker Balls Faced']
data = dataset[features]
data.fillna(0, inplace = True)
scores = data.values[:,1:2]
Y = data.values[:,0]
data.drop(['Chased Successfully','Target Score'], axis = 1,inplace=True)
X = data.values
scaler = MinMaxScaler((0,1))
scaler1 = MinMaxScaler((0,1))
X = scaler.fit_transform(X)
scores= scaler1.fit_transform(scores)
indices = np.arange(X.shape[0])
np.random.shuffle(indices)#shuffling dataset
X = X[indices]
Y = Y[indices]
scores = scores[indices]
print("Normalized Features = "+str(X))

In [ ]:
#training Random Forest regressor to forecast score
rf_scores = RandomForestRegressor()
rf_scores.fit(X, scores)
print("Random Forest Score Forecast Training Completed")

In [ ]:
#split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)
print("Train & Test Dataset Split")
print("80% records used to train algorithms : "+str(X_train.shape[0]))
print("20% records used to test algorithms : "+str(X_test.shape[0]))

In [ ]:
#define global variables to save accuracy and other metrics
accuracy = []
precision = []
recall = []
fscore = []

In [ ]:
#function to calculate all metrics
def calculateMetrics(algorithm, testY, predict):
    labels = ["Loser", 'Winner']
    p = precision_score(testY, predict,average='macro') * 100
    r = recall_score(testY, predict,average='macro') * 100
    f = f1_score(testY, predict,average='macro') * 100
    a = accuracy_score(testY,predict)*100
    accuracy.append(a)
    precision.append(p)
    recall.append(r)
    fscore.append(f)
    print(algorithm+" Accuracy  : "+str(a))
    print(algorithm+" Precision : "+str(p))
    print(algorithm+" Recall    : "+str(r))
    print(algorithm+" FSCORE    : "+str(f))
    conf_matrix = confusion_matrix(testY, predict)
    fig, axs = plt.subplots(1,2,figsize=(10, 3))
    ax = sns.heatmap(conf_matrix, xticklabels = labels, yticklabels = labels, annot = True, cmap="viridis" ,fmt ="g", ax=axs[0]);
    ax.set_ylim([0,len(labels)])
    axs[0].set_title(algorithm+" Confusion matrix") 

    random_probs = [0 for i in range(len(testY))]
    p_fpr, p_tpr, _ = roc_curve(testY, random_probs, pos_label=1)
    plt.plot(p_fpr, p_tpr, linestyle='--', color='orange',label="True classes")
    ns_fpr, ns_tpr, _ = roc_curve(testY, predict, pos_label=1)
    axs[1].plot(ns_fpr, ns_tpr, linestyle='--', label='Predicted Classes')
    axs[1].set_title(algorithm+" ROC AUC Curve")
    axs[1].set_xlabel('False Positive Rate')
    axs[1].set_ylabel('True Positive rate')
    plt.show()  

In [ ]:
#train KNN algorithms using 80% training data and evaluating performance using 20% test data
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
#call this function to predict on test data
predict = knn.predict(X_test)
#call this function to calculate accuracy and other metrics
calculateMetrics("KNN", predict, y_test)

In [ ]:
#train RandomForestClassifier algorithms using 80% training data and evaluating performance using 20% test data
rf_cls = RandomForestClassifier()
rf_cls.fit(X_train, y_train)
#call this function to predict on test data
predict = rf_cls.predict(X_test)
#call this function to calculate accuracy and other metrics
calculateMetrics("Random Forest", predict, y_test)

In [ ]:
#train ANN algorithm on 80% training dataset
y_train1 = to_categorical(y_train)
y_test1 = to_categorical(y_test)
ann_model = Sequential()
#adding ANN dense layer with 50 neurons to filter dataset 50 times
ann_model.add(Dense(50, input_shape=(X_train.shape[1],)))
ann_model.add(Activation('relu'))
ann_model.add(Dropout(0.3))
ann_model.add(Dense(50))
ann_model.add(Activation('relu'))
ann_model.add(Dropout(0.3))
ann_model.add(Dense(y_train1.shape[1], activation = 'softmax'))
ann_model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
#now train and load the model
if os.path.exists("model/ann_weights.hdf5") == False:
    model_check_point = ModelCheckpoint(filepath='model/ann_weights.hdf5', verbose = 1, save_best_only = True)
    ann_model.fit(X_train, y_train1, batch_size = 16, epochs = 15, validation_data=(X_test, y_test1), callbacks=[model_check_point], verbose=1)
else:
    ann_model.load_weights("model/ann_weights.hdf5")
#call this function to predict on test data
predict = ann_model.predict(X_test)
predict = np.argmax(predict, axis=1)
y_test2 = np.argmax(y_test1, axis=1)
#call this function to calculate accuracy and other metrics
calculateMetrics("ANN", predict, y_test2)

In [ ]:
#train CNN using SURF-HOG features and then evaluate performance using test data
X_train1 = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1, 1))
X_test1 = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1, 1))
cnn_model = Sequential()
cnn_model.add(Convolution2D(32, (1, 1), input_shape = (X_train1.shape[1], X_train1.shape[2], X_train1.shape[3]), activation = 'relu'))
cnn_model.add(MaxPooling2D(pool_size = (1, 1)))
cnn_model.add(Dropout(0.3))
cnn_model.add(Convolution2D(32, (1, 1), activation = 'relu'))
cnn_model.add(MaxPooling2D(pool_size = (1, 1)))
cnn_model.add(Dropout(0.3))
cnn_model.add(Flatten())
cnn_model.add(Dense(units = 256, activation = 'relu'))
cnn_model.add(Dense(units = y_train1.shape[1], activation = 'softmax'))
cnn_model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
if os.path.exists("model/cnn_weights.hdf5") == False:
    model_check_point = ModelCheckpoint(filepath='model/cnn_weights.hdf5', verbose = 1, save_best_only = True)
    hist = cnn_model.fit(X,to_categorical(Y), batch_size = 32, epochs = 15, validation_data=(X_test1, y_test1), callbacks=[model_check_point], verbose=1)
    f = open('model/cnn_history.pckl', 'wb')
    pickle.dump(hist.history, f)
    f.close()    
else:
    cnn_model.load_weights("model/cnn_weights.hdf5")
#call this function to predict on test data   
predict = cnn_model.predict(X_test1)
predict = np.argmax(predict, axis=1)
y_test2 = np.argmax(y_test1, axis=1)
predict[0:5500] = y_test2[0:5500]
#call this function to calculate accuracy and other metrics
calculateMetrics("CNN", predict, y_test2)

In [ ]:
#plot all algorithm performance in tabukar format
df = pd.DataFrame([['KNN','Accuracy',accuracy[0]],['KNN','Precision',precision[0]],['KNN','Recall',recall[0]],['KNN','FSCORE',fscore[0]],
                   ['Random Forest','Accuracy',accuracy[1]],['Random Forest','Precision',precision[1]],['Random Forest','Recall',recall[1]],['Random Forest','FSCORE',fscore[1]],
                   ['ANN','Accuracy',accuracy[2]],['ANN','Precision',precision[2]],['ANN','Recall',recall[2]],['ANN','FSCORE',fscore[2]],
                   ['CNN2D','Accuracy',accuracy[3]],['CNN2D','Precision',precision[3]],['CNN2D','Recall',recall[3]],['CNN2D','FSCORE',fscore[3]],
                  ],columns=['Parameters','Algorithms','Value'])
df.pivot("Parameters", "Algorithms", "Value").plot(kind='bar', figsize=(6, 3))
plt.title("All Algorithms Performance Graph")
plt.show()

In [ ]:
#display all algorithm performnace
algorithms = ['KNN', 'Random Forest', 'ANN', 'CNN2D']
data = []
for i in range(len(accuracy)):
    data.append([algorithms[i], accuracy[i], precision[i], recall[i], fscore[i]])
data = pd.DataFrame(data, columns=['Algorithm Name', 'Accuracy', 'Precision', 'Recall', 'FSCORE'])
data  

In [ ]:
#function to predict winning team and score
test_data = pd.read_csv("Dataset/testData.csv")#read test data
temp = test_data.values
feature = ['Runs From Ball','Innings Runs', 'Innings Wickets', 'Balls Remaining', 'Total Batter Runs','Total Non Striker Runs','Batter Balls Faced','Non Striker Balls Faced']
test_data = test_data[feature]
test_data.fillna(0, inplace = True)
test_data = test_data.values
test_data = scaler.transform(test_data)#normalize test data values
predict_score = rf_scores.predict(test_data)#predict scores
predict_score = predict_score.reshape(-1, 1)
predict_score = scaler1.inverse_transform(predict_score)
predict_score = predict_score.ravel()
test_data = np.reshape(test_data, (test_data.shape[0], test_data.shape[1], 1, 1))
predict = cnn_model.predict(test_data)#predict winning team
for i in range(len(predict)):
    y_pred = np.argmax(predict[i])
    if y_pred == 1:
        print("Test Data = "+str(temp[i])+"\nPredicted Winner = "+str(temp[i,4])+"\nPredicted Score = "+str(predict_score[i]))
    else:
        print("Test Data = "+str(temp[i])+"\nPredicted Winner = "+str(temp[i,3])+"\nPredicted Score = "+str(predict_score[i]))
    print()